# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 03c: Data Transformation with Snowpark Connect for Apache Spark

---

### What You'll Learn
- Running PySpark DataFrame transformations **directly on Snowflake warehouse** (no Spark cluster needed)
- Using **Snowpark Connect for Apache Spark** — Snowflake's implementation of Spark Connect protocol
- Building the same CUSTOMER_360, TXN_SUMMARY, and EXECUTIVE_DASHBOARD transformations from notebook 03, but using the Spark DataFrame API
- Saving PySpark results to Snowflake tables

### Why Snowpark Connect?

| Aspect | Traditional Spark | Snowpark Connect |
|--------|------------------|-----------------|
| **Infrastructure** | Manage your own Spark cluster | No cluster — runs on Snowflake warehouse |
| **Data Movement** | ETL data out of Snowflake | Data stays in Snowflake |
| **Governance** | Separate access controls | Snowflake RBAC, masking, lineage |
| **Code Changes** | N/A | Minimal — same PySpark DataFrame API |
| **Scaling** | Manual cluster sizing | Auto-scales with warehouse |

### Two SQL Interfaces

| Interface | Use For |
|-----------|---------|
| `spark.sql()` | Standard SQL (SELECT, CREATE TABLE, INSERT) |
| `SnowflakeSession.sql()` | Snowflake-specific SQL (context functions, time travel, QUALIFY, FLATTEN) |

### Architecture
```
┌─────────────────────────────────────────────────┐
│  Snowflake Notebook (this notebook)             │
│  ┌───────────────────────────────────────────┐  │
│  │  PySpark DataFrame API (client)           │  │
│  │  spark.table("CUSTOMERS").join(...)       │  │
│  └─────────────────┬─────────────────────────┘  │
│                    │ Spark Connect Protocol      │
│  ┌─────────────────▼─────────────────────────┐  │
│  │  Snowflake Warehouse (compute engine)     │  │
│  │  Executes as optimized SQL                │  │
│  └───────────────────────────────────────────┘  │
└─────────────────────────────────────────────────┘
```

> **Reference:** [Snowpark Connect for Apache Spark Documentation](https://docs.snowflake.com/en/developer-guide/snowpark-connect/snowpark-connect-apache-spark)

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

In [ ]:
# =============================================================================
# Step 1: Initialize Snowpark Connect Spark Session
# =============================================================================
# Snowpark Connect for Spark lets you run PySpark code on Snowflake's engine.
# No Spark cluster required — init_spark_session() connects to the warehouse
# that's already attached to this notebook.

from snowflake import snowpark_connect
from snowflake.snowpark_connect.snowflake_session import SnowflakeSession

spark = snowpark_connect.init_spark_session()

# Use SnowflakeSession for Snowflake-specific SQL (context functions, time travel, etc.)
sf = SnowflakeSession(spark)

# Set the database/schema context
sf.use_database("JPMC_CONSUMER_BANKING_HOL")
sf.use_schema("HOL_LAB")

# Verify the connection
sf.sql("SELECT CURRENT_WAREHOUSE() AS warehouse, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema").show()

## Step 2: Read Source Tables into Spark DataFrames

Read the same source tables used in notebook 03 (CUSTOMERS, ACCOUNTS, TRANSACTIONS). With Snowpark Connect, `spark.table()` reads directly from Snowflake — no data movement.

In [ ]:
# =============================================================================
# Step 2: Read source tables as Spark DataFrames
# =============================================================================

customers_df = spark.table("JPMC_CONSUMER_BANKING_HOL.HOL_LAB.CUSTOMERS")
accounts_df = spark.table("JPMC_CONSUMER_BANKING_HOL.HOL_LAB.ACCOUNTS")
transactions_df = spark.table("JPMC_CONSUMER_BANKING_HOL.HOL_LAB.TRANSACTIONS")

# Quick row counts
print(f"CUSTOMERS:    {customers_df.count():,} rows")
print(f"ACCOUNTS:     {accounts_df.count():,} rows")
print(f"TRANSACTIONS: {transactions_df.count():,} rows")

# Preview schema
customers_df.printSchema()

## Step 3: Build CUSTOMER_360 — Unified Customer View

Same transformation as notebook 03's Dynamic Table, but expressed as PySpark DataFrame operations: join CUSTOMERS with ACCOUNTS, then aggregate per customer.

In [ ]:
# =============================================================================
# Step 3: CUSTOMER_360 — Join CUSTOMERS with ACCOUNTS, aggregate
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, sum as spark_sum, count, avg, max as spark_max, get_json_object

# Join customers with accounts
customer_accounts = customers_df.join(
    accounts_df, 
    customers_df["CUSTOMER_ID"] == accounts_df["CUSTOMER_ID"], 
    "left"
)

# Aggregate per customer
customer_360 = customer_accounts.groupBy(
    customers_df["CUSTOMER_ID"],
    col("FIRST_NAME"),
    col("LAST_NAME"),
    col("EMAIL"),
    col("SEGMENT"),
    col("RISK_RATING"),
    customers_df["CREATED_AT"].alias("CUSTOMER_SINCE"),
    col("IS_ACTIVE"),
    get_json_object(col("ADDRESS_JSON"), "$.city").alias("CITY"),
    get_json_object(col("ADDRESS_JSON"), "$.state").alias("STATE")
).agg(
    F.countDistinct(accounts_df["ACCOUNT_ID"]).alias("TOTAL_ACCOUNTS"),
    spark_sum(when(accounts_df["STATUS"] == "ACTIVE", 1).otherwise(0)).alias("ACTIVE_ACCOUNTS"),
    spark_sum(accounts_df["BALANCE"]).alias("TOTAL_BALANCE"),
    avg(accounts_df["BALANCE"]).alias("AVG_ACCOUNT_BALANCE"),
    spark_max(accounts_df["LAST_ACTIVITY_DATE"]).alias("LAST_ACTIVITY"),
    spark_sum(when(accounts_df["ACCOUNT_TYPE"] == "CHECKING", accounts_df["BALANCE"]).otherwise(0)).alias("CHECKING_BALANCE"),
    spark_sum(when(accounts_df["ACCOUNT_TYPE"] == "SAVINGS", accounts_df["BALANCE"]).otherwise(0)).alias("SAVINGS_BALANCE"),
    spark_sum(when(accounts_df["ACCOUNT_TYPE"] == "MONEY_MARKET", accounts_df["BALANCE"]).otherwise(0)).alias("MONEY_MARKET_BALANCE")
)

customer_360.show(5, truncate=False)
print(f"CUSTOMER_360: {customer_360.count():,} rows")

## Step 4: Build TXN_SUMMARY — Daily Transaction Aggregations

Aggregate TRANSACTIONS by day, month, category, channel, and transaction type — same logic as notebook 03's TXN_SUMMARY Dynamic Table.

In [ ]:
# =============================================================================
# Step 4: TXN_SUMMARY — Aggregate transactions by day/month/category/channel
# =============================================================================
from pyspark.sql.functions import to_timestamp, date_trunc, min as spark_min

# Add date columns
txn_with_dates = transactions_df.withColumn(
    "TXN_DAY", date_trunc("DAY", to_timestamp(col("TXN_DATE")))
).withColumn(
    "TXN_MONTH", date_trunc("MONTH", to_timestamp(col("TXN_DATE")))
)

# Aggregate
txn_summary = txn_with_dates.groupBy(
    "ACCOUNT_ID", "TXN_DAY", "TXN_MONTH", "CATEGORY", "CHANNEL", "TXN_TYPE"
).agg(
    count("*").alias("TXN_COUNT"),
    spark_sum("AMOUNT").alias("TOTAL_AMOUNT"),
    avg("AMOUNT").alias("AVG_AMOUNT"),
    spark_max("AMOUNT").alias("MAX_AMOUNT"),
    spark_min("AMOUNT").alias("MIN_AMOUNT"),
    spark_sum(when(col("STATUS") == "COMPLETED", 1).otherwise(0)).alias("COMPLETED_COUNT"),
    spark_sum(when(col("STATUS") == "FAILED", 1).otherwise(0)).alias("FAILED_COUNT"),
    spark_sum(when(col("STATUS") == "REVERSED", 1).otherwise(0)).alias("REVERSED_COUNT"),
    spark_sum(when(col("AMOUNT") > 1000, 1).otherwise(0)).alias("HIGH_VALUE_TXN_COUNT")
)

txn_summary.show(5)
print(f"TXN_SUMMARY: {txn_summary.count():,} rows")

## Step 5: Build EXECUTIVE_DASHBOARD — Top-Level KPIs

Combine CUSTOMER_360 and TXN_SUMMARY for executive-level metrics grouped by state and customer segment.

In [ ]:
# =============================================================================
# Step 5: EXECUTIVE_DASHBOARD — Combine customer + transaction metrics
# =============================================================================
from pyspark.sql.functions import current_date, round as spark_round, countDistinct

# Link TXN_SUMMARY back to customers via ACCOUNTS
# First, get the ACCOUNT_ID → CUSTOMER_ID mapping
account_customer_map = accounts_df.select("ACCOUNT_ID", "CUSTOMER_ID").distinct()

# Join txn_summary with account mapping to get CUSTOMER_ID
txn_with_customer = txn_summary.join(
    account_customer_map, "ACCOUNT_ID", "inner"
).filter(
    col("TXN_MONTH") == date_trunc("MONTH", current_date())
)

# Now join customer_360 with txn_with_customer
executive_dashboard = customer_360.join(
    txn_with_customer,
    customer_360["CUSTOMER_ID"] == txn_with_customer["CUSTOMER_ID"],
    "left"
).groupBy(
    customer_360["STATE"],
    customer_360["SEGMENT"]
).agg(
    countDistinct(customer_360["CUSTOMER_ID"]).alias("TOTAL_CUSTOMERS"),
    spark_sum(when(customer_360["IS_ACTIVE"] == True, 1).otherwise(0)).alias("ACTIVE_CUSTOMERS"),
    avg(customer_360["TOTAL_BALANCE"]).alias("AVG_CUSTOMER_BALANCE"),
    spark_sum(customer_360["TOTAL_BALANCE"]).alias("TOTAL_DEPOSITS"),
    spark_sum("TXN_COUNT").alias("MONTHLY_TXN_COUNT"),
    spark_sum("TOTAL_AMOUNT").alias("MONTHLY_TXN_VOLUME"),
    avg("AVG_AMOUNT").alias("AVG_TXN_SIZE"),
    spark_sum("FAILED_COUNT").alias("MONTHLY_FAILED_TXNS"),
    spark_sum("HIGH_VALUE_TXN_COUNT").alias("HIGH_VALUE_TXNS")
)

executive_dashboard.show(10)
print(f"EXECUTIVE_DASHBOARD: {executive_dashboard.count():,} rows")

## Step 6: Save Results to Snowflake Tables

Write the PySpark DataFrames back to Snowflake as tables. Snowpark Connect's `saveAsTable` creates native Snowflake tables directly.

In [ ]:
# =============================================================================
# Step 6: Write results to Snowflake tables
# =============================================================================

# Save CUSTOMER_360
customer_360.write.mode("overwrite").saveAsTable("JPMC_CONSUMER_BANKING_HOL.HOL_LAB.CUSTOMER_360_SPARK")

# Save TXN_SUMMARY
txn_summary.write.mode("overwrite").saveAsTable("JPMC_CONSUMER_BANKING_HOL.HOL_LAB.TXN_SUMMARY_SPARK")

# Save EXECUTIVE_DASHBOARD
executive_dashboard.write.mode("overwrite").saveAsTable("JPMC_CONSUMER_BANKING_HOL.HOL_LAB.EXECUTIVE_DASHBOARD_SPARK")

print("All 3 tables saved to Snowflake!")

## Step 7: Validate Results with Spark SQL

Use `spark.sql()` to run validation queries — these execute as native SQL on Snowflake.

In [ ]:
# =============================================================================
# Step 7: Validate — run SQL queries via SnowflakeSession
# =============================================================================

# Row counts for all output tables
sf.sql("""
    SELECT 'CUSTOMER_360_SPARK' AS table_name, COUNT(*) AS row_count FROM CUSTOMER_360_SPARK
    UNION ALL
    SELECT 'TXN_SUMMARY_SPARK', COUNT(*) FROM TXN_SUMMARY_SPARK
    UNION ALL
    SELECT 'EXECUTIVE_DASHBOARD_SPARK', COUNT(*) FROM EXECUTIVE_DASHBOARD_SPARK
""").show()

# Top 5 customers by total balance
sf.sql("""
    SELECT CUSTOMER_ID, FIRST_NAME, LAST_NAME, SEGMENT, TOTAL_BALANCE, TOTAL_ACCOUNTS
    FROM CUSTOMER_360_SPARK
    ORDER BY TOTAL_BALANCE DESC
    LIMIT 5
""").show()

## Step 8: UDF Demo — Custom Business Logic in PySpark

Snowpark Connect supports PySpark UDFs that execute on Snowflake's engine. This example creates a risk scoring function.

In [ ]:
# =============================================================================
# Step 8: PySpark UDF — Custom risk scoring logic
# =============================================================================
from pyspark.sql.connect.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def calculate_risk_tier(total_balance: float, active_accounts: int, segment: str) -> str:
    """Multi-factor risk assessment based on banking heuristics"""
    score = 0
    
    # Balance factor
    if total_balance and total_balance > 100000:
        score += 3
    elif total_balance and total_balance > 25000:
        score += 2
    elif total_balance and total_balance > 5000:
        score += 1
    
    # Account diversification factor
    if active_accounts and active_accounts >= 3:
        score += 2
    elif active_accounts and active_accounts >= 2:
        score += 1
    
    # Segment factor
    if segment == "WEALTH":
        score += 2
    elif segment == "PREMIUM":
        score += 1
    
    # Tier assignment
    if score >= 6:
        return "LOW_RISK"
    elif score >= 3:
        return "MODERATE_RISK"
    else:
        return "HIGH_RISK"

# Apply UDF to CUSTOMER_360
risk_scored = customer_360.withColumn(
    "RISK_TIER",
    calculate_risk_tier(col("TOTAL_BALANCE"), col("ACTIVE_ACCOUNTS"), col("SEGMENT"))
)

# Show risk distribution
risk_scored.groupBy("RISK_TIER").count().orderBy("count", ascending=False).show()

## Step 9: Compare Approaches — Dynamic Tables vs Snowpark Connect

Both approaches produce the same results. Here's when to use each:

In [ ]:
# =============================================================================
# Step 9: Compare row counts between DT approach and Snowpark Connect approach
# =============================================================================

sf.sql("""
    SELECT 
        'CUSTOMER_360 (Dynamic Table)' AS source, COUNT(*) AS row_count FROM CUSTOMER_360
    UNION ALL
    SELECT 
        'CUSTOMER_360_SPARK (Snowpark Connect)', COUNT(*) FROM CUSTOMER_360_SPARK
    UNION ALL
    SELECT 
        'TXN_SUMMARY (Dynamic Table)', COUNT(*) FROM TXN_SUMMARY
    UNION ALL
    SELECT 
        'TXN_SUMMARY_SPARK (Snowpark Connect)', COUNT(*) FROM TXN_SUMMARY_SPARK
""").show(truncate=False)

## Summary

### Snowpark Connect for Apache Spark — Key Takeaways

| Feature | Details |
|---------|---------|
| **No Spark Cluster** | All compute runs on your Snowflake warehouse |
| **Same API** | Standard PySpark DataFrame + Spark SQL — minimal migration |
| **UDF Support** | Python UDFs execute on Snowflake engine |
| **Governance** | Full Snowflake RBAC, masking, lineage applies |
| **saveAsTable** | Write DataFrames directly to Snowflake tables |

### When to Use Each Approach

| Use Case | Best Approach |
|----------|--------------|
| Automated, always-fresh transformations | **Dynamic Tables** (notebook 03) |
| Existing PySpark code migration | **Snowpark Connect** (this notebook) |
| Complex Python/ML logic with Spark | **Snowpark Connect** |
| Simple SQL-only pipelines | **Dynamic Tables** |
| Ad-hoc exploration with Spark semantics | **Snowpark Connect** |

---

**Next:** Proceed to `03b_Stored_Procedures_Tasks.ipynb` for imperative transformations with Stored Procedures + Tasks.